In [9]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
sys.path.insert(0, "..")
from importlib import reload
import matplotlib.pyplot as plt
import equations as eq
reload (eq);
import trajectory_lib as tr
reload (tr);

participant = '3NA'
motion_list  =  ['All_motions']
GH_seq = 'YZY'
include_q_clavicula_init = True

motion_folder = motion_list[0]
motion_name = motion_list[0]

OS_struct = sc.io.loadmat('../Motions/'+participant+'/OS_model.mat')
# optim_params_struct = sc.io.loadmat('../Motions/'+participant+'/'+motion_name+'/params_optimized_'+motion_name+'.mat')
include_activation_dynamics = False

act_w = 1
vel_w = 1

MM,FO,q,u,faux,fr,frstar,kinematical,xdot,first_elips_scale,elips_trans = eq.create_eoms_quat_w_RF(OS_struct,weight = 0,derive = 'numeric',gen_matlab_functions = 0)
# TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,optim_params = 'group', derive = 'numeric')

[0.13665370273077132, 0.6179200122197827, 1.7974341835466003, 0.47863835366726576, 0.47863835366726576, 0.4598922688054804]


In [10]:
OS_struct = sc.io.loadmat('../Motions/'+participant+'/OS_model.mat')

In [11]:
# # params_init = {**fmax_init, **lceopt_init}
# params_init = fmax_init
# # params_range = {**fmax_range, **lceopt_range}
# params_range = fmax_range
# myKeys = list(params_init.keys())
# myKeys.sort()

# # Sorted Dictionary
# sd = {i: params_init[i] for i in myKeys}
# sorted_params_init = np.array([*sd.values()])
# num_params = len(sorted_params_init)
# print(sd)

In [12]:
# reload(eq)
# eq.optimize_parameters(params_init,mus_groups)

In [ ]:
reload(eq)
reload(tr)
clav_pos = 0.4

weights = [9,9,9,10,11,9]
wGHs = [1,0,5,1,1]
tilt_ys = [15]
tilt_zs = [-10,-10,-10,-10,-10,-10]
w_rxs = [0.1,0.05,0.01,0.05,0.05,0.1]
# diff_weights = [1]
w_optim_params = 1
name_w_param = 'e1'
simulations = ['All_motions']  # 0 use optimized, 1 optimize
w_optim_fmax = 1
w_optim_lceopt = 1
isim = 0
for tilt_y in tilt_ys:
    for tilt_z in tilt_zs:
        iweight = weights[isim]
        # iweight = 9
        sim = 'All_motions'
        wGH = wGHs[isim]
        # wGH = 1
        w_rx = w_rxs[isim]
        if sim == 'optimize':
            TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,optim_params = 'group', derive = 'numeric')
            emg_name = 'all'
            params_init = {**fmax_init, **lceopt_init}
            params_range = {**fmax_range, **lceopt_range}
            myKeys = list(params_init.keys())
            myKeys.sort()
            sd = {i: params_init[i] for i in myKeys}
            sorted_params_init = np.array([*sd.values()])
            num_params = len(sorted_params_init)
            print(myKeys)
            print('num_params = ', num_params)
            include_activation_dynamics = False
            optimize = 1

        # if opt == 'op':
        #     optim_params_struct = sc.io.loadmat('../Motions/'+participant+'/'+motion_name+'/params_optimized_'+motion_name+'_'+str(weights[iweight])+'_'+name_w_param+'.mat')
        #     emg_name = 'all'

        #     TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,optim_params = optim_params_struct, derive = 'numeric')
        #     num_params = 0
        #     include_activation_dynamics = False

        else:
            motion_folder = sim
            emg_name = sim
            motion_list = [sim]
            motion_name = sim
            # optim_params_struct = sc.io.loadmat('../Motions/'+participant+'/All_motions/params_optimized_All_motions_'+str(weights[iweight])+'_'+name_w_param+'.mat')
            optim_params_struct = sc.io.loadmat('../Motions/'+participant+'/All_motions/params_optimized_All_motions_'+str(iweight)+'_'+name_w_param+'.mat')

            TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,optim_params = optim_params_struct, derive = 'numeric')
            num_params = 0
            include_activation_dynamics = False
            optimize = 0

        

        if include_q_clavicula_init:
            q_clavicula_init = sc.io.loadmat('../Motions/'+participant+'/'+motion_list[0]+'/res_quat_'+motion_list[0]+'_init.mat')['data']['q_clavicula_init']
            # q_clavicula_init = sc.io.loadmat('../Motions/'+participant+'/'+motion_name+'/res_quat_'+motion_name+'_184.mat')['data']['q_clavicula_init']
            # q_clavicula_init = sc.io.loadmat('../Motions/'+participant+'/'+motion_list[0]+'/res_quat_Scabduction_scaling_234.mat')['data']['q_clavicula_init']

        weight = 301
        struct_name = 'res_quat_'+motion_list[0]+'_'+str(int(weight))
        # GH_spring_torque = eq.joint_spring_quat(q[8:12],sp.Matrix([1,0,0,0]))
        # spring_torques = sp.Matrix([0,0,0]).col_join(-GH_spring_torque).col_join(GH_spring_torque).col_join(sp.Matrix([0]))
        eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+sp.Matrix([TE+sp.Matrix(TE_conoid)]).col_join(GH_mus_forces))
        # eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+sp.Matrix([TE+sp.Matrix(TE_conoid)]).col_join(sp.Matrix([0,0,0])))
        traj_w = weight

        if include_activation_dynamics:
            excitations = []
            act_ode = []
            for i in range(len(activations)):
                excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
                current_mus_ind = int(str(activations[i])[4:-3])
                current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
                t_act = current_mus['tact'][0,0].item()
                t_deact = current_mus['tdeact'][0,0].item()
                act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i]))
            # print('act_ode = ', act_ode)
            sp_act_ode = sp.Matrix(act_ode)
            eoms_implicit = eoms_implicit.col_join(sp_act_ode)

        reload(eq)
        num_nodes = 201
        file = '../Motions/' + participant + '/' + motion_folder + '/' + motion_name
        traj_original, interval_value, time = tr.exp_trajectory_quat(file,num_nodes)
        traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos)
        emg, indexes = tr.exp_emg('../Motions/'+participant+'/'+motion_folder+'/EMG_'+participant+'_'+emg_name+'.mat', num_nodes = num_nodes,EMG_num = 1)

        if include_activation_dynamics:
            state_symbols = tuple(q+u+faux+activations)
            specified_symbols = tuple(excitations)
        else:
            state_symbols = tuple(q+u+faux)
            specified_symbols = tuple(activations)

        num_states = len(state_symbols) 
        num_q = len(q)
        num_u = len(u)
        num_faux = len(faux)
        num_inputs = len(specified_symbols)
        t = me.dynamicsymbols._t
        objective_traj,objective_traj_jac = eq.custom_objective_quat(num_q,interval_value,clav_pos,True)

        objective_act,objective_act_jac = eq.min_activation(activations,interval_value)
        objective_exc,objective_exc_jac = eq.min_activation(activations,interval_value)
        objective_maxstab, objective_maxstab_jac = eq.max_GH_stab(3,tilt_y=tilt_y,tilt_z=tilt_z,w_rx=w_rx)

        obj_min_diff,obj_min_diff_jac = eq.objective_activation_diff(num_nodes, interval_value)

        if optimize == 1:
            obj_opt_params, obj_opt_params_jac = eq.optimize_parameters(num_params,mus_groups,'group',w_optim_fmax,w_optim_lceopt)
        
        w_diff_vel = 1
        w_diff_act = 0
        w_diff_exc = 1e-1
        node1 = 0
        node2 = num_nodes//2
        node3 = num_nodes-1

        w_min_squared_act = 1
        w_min_squared_exc = 0
        
        if optimize == 1:
            if include_activation_dynamics:
                w_min_emg_act = 0
                w_min_emg_exc = weights[iweight] 
            else: 
                w_min_emg_act = weights[iweight]
                w_min_emg_exc = 0
        else:
            w_min_emg_act = 0
            w_min_emg_exc = 0


        def obj(free):
            # min_traj = traj_w * interval_value * np.sum((traj_original.flatten() - free[:10*num_nodes])**2)
            min_traj = traj_w * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj))
            # min_traj = traj_w * (objective_traj(np.array(np.split(free[:num_q*num_nodes],num_q))[:,node1],traj[:,node1]) + objective_traj(np.array(np.split(free[:num_q*num_nodes],num_q))[:,node2],traj[:,node2]) + objective_traj(np.array(np.split(free[:num_q*num_nodes],num_q))[:,node3],traj[:,node3]))
            # min_traj_lastnode = 3*traj_w*objective_traj(np.array(np.split(free[:num_q*num_nodes],num_q))[:,node3],traj[:,node3])

            min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))

            # min_torque = act_w * interval_value * np.sum(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes]**2)
            min_torque = act_w * np.sum(objective_act(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs),emg,indexes,w_min_squared_act,w_min_emg_act))

            min_instab = wGH * np.sum(objective_maxstab(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

            if optimize == 0:
                opt_params = 0
            else:
                opt_params = w_optim_params * obj_opt_params(*free[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes])
                # opt_params = 0
            obj = (min_traj + min_torque+ min_vel_dif + opt_params + min_instab) #   
            # min_act_dif = w_diff_act * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))))
            if include_activation_dynamics:
                min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))))
                min_exc = act_w * np.sum(objective_exc(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs),emg,indexes,w_min_squared_exc,w_min_emg_exc))
                obj += (min_exc_dif + min_exc)

            return obj.item()

        def obj_grad(free):
            grad = np.zeros_like(free)
            # grad[:10*num_nodes] = traj_w * 2.0 * interval_value * (free[:10*num_nodes] - traj_original.flatten())
            grad[:num_q*num_nodes] += traj_w * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj))

            # grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] = act_w * 2.0 * interval_value * free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] #+ w_diff_act * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))[0,:,:])))
            grad[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes] += act_w * np.concatenate(objective_act_jac(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs),emg,indexes,w_min_squared_act,w_min_emg_act))
            # print(np.shape(objective_exc_jac(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs),emg,indexes)))

            grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))
            # print(objective_maxstab_jac(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))
            grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes] += wGH * np.concatenate(objective_maxstab_jac(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

            if optimize == 1:
                grad[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes] += w_optim_params * obj_opt_params_jac(*free[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes])[0]

            if include_activation_dynamics:
                grad[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))[0,:,:])))
                grad[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes] += np.concatenate(objective_exc_jac(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs),emg,indexes,w_min_squared_exc,w_min_emg_exc))

            
            # ## reach ##
            # first_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:num_q*num_nodes],num_q))[:,node1],traj[:,node1])))
            # second_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:num_q*num_nodes],num_q))[:,node2],traj[:,node2])))
            # third_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:num_q*num_nodes],num_q))[:,node3],traj[:,node3])))
            # # print(np.shape(first_grad_vals))
            # grad_traj = np.zeros((num_nodes,num_q))
            # grad_traj[node1,:] = first_grad_vals
            # grad_traj[node2,:] = second_grad_vals
            # grad_traj[node3,:] = third_grad_vals
            # grad[:num_q*num_nodes] += 3*traj_w * np.concatenate(grad_traj.T)
            # ## end reach ##

            return grad
        
        print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes + num_params)))
        print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes + num_params))))
        instance_constraints = []
        # for i in range(13):
        if include_q_clavicula_init:
            instance_constraints.append(state_symbols[0].func(0.0)-q_clavicula_init[0,0][0,0])
            instance_constraints.append(state_symbols[1].func(0.0)-q_clavicula_init[0,0][0,1])
            instance_constraints.append(state_symbols[2].func(0.0)-q_clavicula_init[0,0][0,2])
            instance_constraints.append(state_symbols[3].func(0.0)-q_clavicula_init[0,0][0,3])
            # instance_constraints.append(state_symbols[4].func(0.0)-q_clavicula_init[0,0][0,4])
            # instance_constraints.append(state_symbols[5].func(0.0)-q_clavicula_init[0,0][0,5])
            # instance_constraints.append(state_symbols[6].func(0.0)-q_clavicula_init[0,0][0,6])
            # instance_constraints.append(state_symbols[7].func(0.0)-q_clavicula_init[0,0][0,7])
            # instance_constraints.append(state_symbols[0].func(time[-1])-q_clavicula_init[0,0][0,0])
            # instance_constraints.append(state_symbols[1].func(time[-1])-q_clavicula_init[0,0][0,1])
            # instance_constraints.append(state_symbols[2].func(time[-1])-q_clavicula_init[0,0][0,2])
            # instance_constraints.append(state_symbols[3].func(time[-1])-q_clavicula_init[0,0][0,3])
        else:
            instance_constraints.append(state_symbols[0].func(0)**2 + state_symbols[1].func(0)**2 + state_symbols[2].func(0)**2 + state_symbols[3].func(0)**2 - 1) # SC
        instance_constraints.append(state_symbols[4].func(0)**2 + state_symbols[5].func(0)**2 + state_symbols[6].func(0)**2 + state_symbols[7].func(0)**2 - 1) # AC
        instance_constraints.append(state_symbols[8].func(0)**2 + state_symbols[9].func(0)**2 + state_symbols[10].func(0)**2 + state_symbols[11].func(0)**2 - 1) # GH

        # instance_constraints.append(state_symbols[-1].func(0.0)-0) 
            
        bounds1 = (0.0,1.0)
        bounds = (bounds1,)*len(activations)
        bndrs = dict(zip(activations,bounds))
        if include_activation_dynamics:
            bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
            bndrs.update(bndrs_exc)
        
        # supra, infra, subscap
        # act_dmg = ['act_54(t)', 'act_55(t)', 'act_56(t)', 'act_57(t)', 'act_58(t)', 'act_59(t)', 'act_67(t)', 'act_68(t)', 'act_69(t)', 'act_70(t)', 'act_71(t)', 'act_72(t)', 'act_73(t)', 'act_74(t)', 'act_75(t)', 'act_76(t)', 'act_77(t)', 'act_78(t)', 'act_79(t)', 'act_80(t)', 'act_81(t)'] # ]
        
        # supra, infra        
        act_dmg = ['act_54(t)', 'act_55(t)', 'act_56(t)', 'act_57(t)', 'act_58(t)', 'act_59(t)', 'act_67(t)', 'act_68(t)', 'act_69(t)', 'act_70(t)'] # ]

        for iact in activations:
            if str(iact) in act_dmg:
                # if divisible by 2
                # if isim % 2:
                #     # print('divisibile')
                #     bndrs.update({iact: (0.0, 1.0)})
                # else:
                if isim == 2:
                    bndrs.update({iact: (0.0, 0.02)})
                    # print('tear')
                else:
                    bndrs.update({iact: (0.0, 1.0)})
                    

        # SCq_bndr = {q[0] : (0.75, 1),
        #             q[1] : (-0.1, 0.4),
        #             q[2] : (-0.5, -0.3),
        #             q[3] : (-0.1, 0.35)}
        # bndrs.update(SCq_bndr)
        # ACq_bndr = {q[4] : (0.8, 1),
        #             q[5] : (0.1, 0.3),}
        #             # q[6] : (0.3, 0.65),
        #             # q[7] : (-0.15, 0.15)}
        # bndrs.update(ACq_bndr)
        # GHq_bndr = {q[8] : (0.8, 1),
        #             q[12]: (0*np.pi/180, 150*np.pi/180)}
        # bndrs.update(GHq_bndr)
        for i in range(num_q):
            if i == 8:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.12, 1.0)})
            else:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.12, max(traj_original[i,:])+0.12)})

        if optimize == 1:
            bndrs.update(params_range)
        print(bndrs)



        start = tm.time()
        prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                    num_nodes, interval_value,
                    known_parameter_map={},
                    instance_constraints=instance_constraints,
                    bounds=bndrs,
                    integration_method='midpoint',
                    parallel = False)


        time_to_create = tm.time() - start
        print(time_to_create)

        prob.add_option('limited_memory_max_history', 40)
        initial_guess = np.ones(prob.num_free)*0.0
        # initial_guess[:13*num_nodes] = traj_original.flatten()
        # initial_guess[:(num_states+num_inputs)*num_nodes] = tr.initial_guess_from_solution('../Motions/'+participant+'/'+motion_folder+'/res_quat_All_motions_180_use_optimized_14_nostab.mat',prob.num_free)
 
        time_2_solve_start = tm.time()

        prob.add_option('max_iter',2000)
        initial_guess[:13*num_nodes] = traj_original.flatten()
        initial_guess[(num_q + num_u)*num_nodes:(num_q + num_u + 1)*num_nodes] = -0.75
        initial_guess = tr.initial_guess_from_solution('../Motions/'+participant+'/'+motion_folder+'/res_quat_All_motions_optim.mat',prob.num_free)[0:prob.num_free]

        # initial_guess[0:(num_q + num_u + num_inputs)*num_nodes] = tr.initial_guess_from_solution('../Motions/'+participant+'/'+motion_folder+'/res_quat_All_motions_204.mat',prob.num_free)[0:prob.num_free]
        


        if optimize == 1:
            initial_guess[(num_states+num_inputs)*num_nodes:(num_states+num_inputs+num_params)*num_nodes] = np.ones(num_params)
            
        solution, info = prob.solve(initial_guess)
        time_2_solve = tm.time() - time_2_solve_start
        print(info['status_msg'])
        print(info['obj_val'])
        act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
        objective_value = prob.obj_value
        print('Objective activations: ', act_obj)

        reload(tr)
        if optimize == 1:
            file_name = '../Motions/'+participant+'/'+motion_folder+'/' + struct_name + '_optimize_'+str(weights[iweight])+'.mat'

        else:
            # file_name = '../Motions/'+participant+'/'+motion_folder+'/' + struct_name+str(weights[iweight])+str(isim)+'.mat'
            file_name = '../Motions/'+participant+'/'+motion_folder+'/' + struct_name+str(14)+str(isim)+'.mat'

        tr.sol2struct(solution,activations,num_q,num_u,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

        file_name_mot = '../Motions/'+participant+'/'+motion_folder+'/' + struct_name + str(isim) + '.mot'
        tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot, GH_seq)

        if optimize == 1:
            optimized_params = dict(zip(myKeys, solution[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes]))
            sc.io.savemat('../Motions/'+participant+'/'+motion_folder+'/params_optimized_'+motion_name+'_'+str(weights[iweight])+'_'+name_w_param+'.mat', {'optimized_params': optimized_params})
        prob.close()
        isim += 1

[np.float64(0.0972191888386686), np.float64(0.08080084773248851), np.float64(0.0762590124878112), np.float64(0.07585661671812728), np.float64(0.07362553975582942), np.float64(0.07115541883328538), np.float64(0.0691633858312337), np.float64(0.06135461646319119), np.float64(0.0805578146029692), np.float64(0.10071718858373203), np.float64(0.10796818871120008), np.float64(0.1095706685928505), np.float64(0.11899077132255263), np.float64(0.16066410172547338), np.float64(0.13256315617653128), np.float64(0.07026290146588043), np.float64(0.09974576600254396), np.float64(0.09873544977231563), np.float64(0.12684061035866778), np.float64(0.10247017762553748), np.float64(0.10972117775300551), np.float64(0.12422317800794165), np.float64(0.12486062856859818), np.float64(0.12509967252884435), np.float64(0.09057104714976466), np.float64(0.10926160159058962), np.float64(0.108849867409808), np.float64(0.10613275743641098), np.float64(0.10366263552572855), np.float64(0.10045147524830636), np.float64(0.093

IndexError: list index out of range

In [ ]:
# fig, axes = plt.subplots(26, 1, sharex=True,
#                          figsize=(6.4, 0.8*30),
#                          layout='compressed')
# prob.plot_trajectories(solution, axes=axes)